In [ ]:
%pip install -q kagglehub librosa seaborn scikit-learn
import warnings, pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, librosa.display, kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings("ignore")
SR, SEGMENTS, N_MFCC, EPOCHS, SEED = 22050, 10, 13, 40, 42
SEG_LEN = SR * 30 // SEGMENTS
np.random.seed(SEED)

In [ ]:
# Download dataset
DATA = Path(kagglehub.dataset_download(
    "andradaolteanu/gtzan-dataset-music-genre-classification"
))
matches = list(DATA.rglob("genres_original"))
GENRES_DIR = matches[0] if matches else DATA / "Data" / "genres_original"
if not GENRES_DIR.is_dir():
    raise FileNotFoundError("genres_original folder not found.")
genres = sorted(p.name for p in GENRES_DIR.iterdir() if p.is_dir())
print("Genres:", genres)

In [ ]:
# Extract MFCC features
def extract_mfcc(path):
    try:
        signal, _ = librosa.load(str(path), sr=SR, duration=30)
        return [
            librosa.feature.mfcc(
                y=signal[i*SEG_LEN:(i+1)*SEG_LEN],
                sr=SR, n_mfcc=N_MFCC
            ).T
            for i in range(SEGMENTS)
            if len(signal[i*SEG_LEN:(i+1)*SEG_LEN]) == SEG_LEN
        ]
    except Exception:
        return []

In [ ]:
# Load audio files
files, labels = [], []
for i, genre in enumerate(genres):
    for f in sorted((GENRES_DIR / genre).glob("*.wav")):
        files.append(f)
        labels.append(i)
print("Total songs:", len(files))
f_train, f_test, l_train, l_test = train_test_split(
    files, labels, test_size=0.2, stratify=labels, random_state=SEED
)

In [ ]:
# Prepare train and test data
def prepare_data(paths, targets):
    X, y = [], []
    for path, label in zip(paths, targets):
        features = extract_mfcc(path)
        X.extend(features)
        y.extend([label] * len(features))
    return np.array(X), np.array(y)
X_train, y_train = prepare_data(f_train, l_train)
X_test, y_test = prepare_data(f_test, l_test)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(len(X_train), -1)).reshape(X_train.shape)
X_test = scaler.transform(X_test.reshape(len(X_test), -1)).reshape(X_test.shape)

In [ ]:
# Display MFCC
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
librosa.display.specshow(X_train[0].T, sr=SR, x_axis="time")
plt.title("MFCC Features")
plt.subplot(1, 2, 2)
librosa.display.specshow(X_train[0].T, sr=SR, x_axis="time")
plt.title("Normalized MFCC")
plt.tight_layout()
plt.show()

In [ ]:
# Build LSTM model
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1], N_MFCC)),
    layers.LSTM(128, return_sequences=True),
    layers.LSTM(64),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(genres), activation="softmax")
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
]

In [ ]:
# Train model
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=64,
    callbacks=callbacks,
    verbose=2
)

In [ ]:
# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Loss: {loss:.4f}")

In [ ]:
# Accuracy and loss graphs
plt.figure(figsize=(10, 4))
for i, metric in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, i+1)
    plt.plot(history.history[metric], label="Training")
    plt.plot(history.history["val_"+metric], label="Validation")
    plt.title(metric.capitalize())
    plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Classification report
predictions = model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, predictions,
                            target_names=genres, zero_division=0))

In [ ]:
# Confusion matrix
plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(y_test, predictions),
            annot=True, fmt="d", cmap="Blues",
            xticklabels=genres, yticklabels=genres)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Music Genre Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Predict genre of a new song
def predict_song(path):
    features = extract_mfcc(path)
    if not features:
        return "Unreadable", 0.0
    X = np.array(features)
    X = scaler.transform(X.reshape(len(X), -1)).reshape(X.shape)
    probs = model.predict(X, verbose=0).mean(axis=0)
    return genres[probs.argmax()], float(probs.max())

In [ ]:
# Test sample songs
for path, label in list(zip(f_test, l_test))[:5]:
    genre, confidence = predict_song(path)
    print(f"{path.name}: Actual={genres[label]}, Predicted={genre}, Confidence={confidence:.2f}")

In [ ]:
# Save model and scaler
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
model.save(out_dir / "gtzan_lstm.keras")
with open(out_dir / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("Model and scaler saved successfully in outputs/")